# SimThink D: Decision Models that Fit in One Game Tick

Make fast, tiny decision models from your own rules - no training data, no cloud, about 1 ms per call on one CPU core.

This notebook walks through the key ideas:
- **Decider**: A trained model that picks one action from a situation sentence (in ~1 ms).
- **Teacher**: A function that writes the rules you want the decider to learn.
- **Tick**: One frame in a game (e.g., 1/35 second at 35 Hz). Your decision must fit inside.

In [ ]:
# Install simthinkd from PyPI (takes about 20 seconds).
!pip install "simthinkd[train]"

## One Decision in 3 Lines

Load a pre-trained **Decider** ("doom-defend"), give it a situation sentence, get back the best action, confidence, and how long it took.

In [ ]:
from simthinkd import Decider

d = Decider("doom-defend")
result = d.decide("seen: Demon left a30 d5 | enemies 1 | sway left | gun ready | ammo25")
print(result)

## Speed: 1,000 Decisions

Run 1,000 fast decisions and show median time and p95 (95th percentile). At 35 Hz, one tick is 28.6 ms. If most decisions fit inside 28.6 ms, they fit inside the game frame.

In [ ]:
import statistics
from simthinkd import Decider

d = Decider("doom-defend")
times = []

for i in range(1000):
    result = d.decide("seen: Demon left a30 d5 | enemies 1 | sway left | gun ready | ammo25")
    times.append(result.ms)

times_sorted = sorted(times)
median = statistics.median(times_sorted)
p95 = times_sorted[int(0.95 * len(times_sorted))]
tick_ms = 1000.0 / 35.0
within_tick = sum(1 for t in times if t <= tick_ms) / len(times)

print(f"Median: {median:.2f} ms")
print(f"P95: {p95:.2f} ms")
print(f"One tick at 35 Hz: {tick_ms:.1f} ms")
print(f"Decisions within one tick: {within_tick:.1%}")

## Your Own Rules in 8 Seconds

Write a tiny teacher (a function that says what action to take for each situation), generate examples, and train a decider. This example is a simple thermostat: maintain temperature in a comfort zone.

In [ ]:
import random
import simthinkd

THERMOSTAT_ACTIONS = {
    'OFF': 'Turn off heating and cooling.',
    'HEAT': 'Turn on the heater.',
    'COOL': 'Turn on the air conditioner.',
}
THERMOSTAT_GOAL = 'Keep the room at a comfortable temperature between 20C and 24C.'

def thermostat_teacher(state):
    temp = state['temp']
    setpoint = state['setpoint']
    if temp < setpoint - 1:
        return 'HEAT'
    elif temp > setpoint + 1:
        return 'COOL'
    else:
        return 'OFF'

rng = random.Random(42)
examples = []
for _ in range(3000):
    temp = rng.uniform(15, 30)
    setpoint = rng.choice([20, 21, 22, 23, 24])
    state = {'temp': temp, 'setpoint': setpoint}
    action = thermostat_teacher(state)
    text = f"temp {temp:.1f}C | setpoint {setpoint}C"
    examples.append((text, action))

print(f"Generated {len(examples)} examples.")

d = simthinkd.fit(
    examples,
    THERMOSTAT_ACTIONS,
    goal=THERMOSTAT_GOAL,
    out="thermostat",
    quiet=True,
)
print("Trained decider saved to thermostat folder.")

test_cases = [
    "temp 18.0C | setpoint 22C",
    "temp 25.0C | setpoint 22C",
    "temp 21.5C | setpoint 22C",
]
print("\nTest predictions:")
for case in test_cases:
    result = d.decide(case)
    print(f"  {case:30s} -> {result.choice} ({result.confidence:.0%})")

## Benchmark on Real Doom States

Measure how fast and accurate the bundled doom-defend decider is on 1,050 recorded game states from ViZDoom's defend_the_center scenario.

In [ ]:
from simthinkd import bench

result = bench.run()
print(bench.report(result))

## Serve Over HTTP and Call It

Start a simple HTTP server in a background thread and make decisions over the network. This is how you'd integrate the decider into a web app or a remote game engine.

In [ ]:
import json
import threading
import time
import urllib.request
from http.server import HTTPServer, BaseHTTPRequestHandler

from simthinkd import Decider

DECIDER = Decider("doom-defend")

class DecisionHandler(BaseHTTPRequestHandler):
    def do_POST(self):
        if self.path != '/v1/systemone':
            self.send_response(404)
            self.end_headers()
            return
        
        content_length = int(self.headers.get('Content-Length', 0))
        body = json.loads(self.rfile.read(content_length).decode())
        answers = DECIDER.predict(body)
        
        self.send_response(200)
        self.send_header('Content-Type', 'application/json')
        self.end_headers()
        self.wfile.write(json.dumps({'answers': answers}).encode())
    
    def log_message(self, format, *args):
        pass

server = HTTPServer(('127.0.0.1', 8000), DecisionHandler)
thread = threading.Thread(target=server.serve_forever, daemon=True)
thread.start()
print("Server started at http://127.0.0.1:8000")
time.sleep(0.5)

request = DECIDER.request("seen: Demon left a30 d5 | enemies 1 | sway left | gun ready | ammo25")
req = urllib.request.Request(
    'http://127.0.0.1:8000/v1/systemone',
    data=json.dumps(request).encode(),
    headers={'Content-Type': 'application/json'}
)
with urllib.request.urlopen(req) as response:
    reply = json.load(response)
    choice = reply['answers']['operation']['choice']
    conf = reply['answers']['operation']['confidence']
    print(f"HTTP decision: {choice} ({conf:.0%})")

server.shutdown()
print("Server stopped.")

## Next Steps

1. **Try other presets**: `Decider("doom-corridor")` for a different Doom scenario.
2. **Write your own teacher**: Replace `thermostat_teacher` with rules from your game or app.
3. **Tune training**: Increase `steps` in `simthinkd.fit()` for better accuracy, or adjust `goal` to change what the decider optimizes for.
4. **Deploy**: Use the HTTP server pattern (cell 6) to run the decider in production.
5. **Benchmark**: Use `bench.run()` on your own domain to measure speed and accuracy.

For details, see the [SimThink D docs](https://github.com/MSSJ-AI-ORG/simthinkd).